<a href="https://colab.research.google.com/github/GhalaDev/AI-BoardGame-Forge/blob/main/AI_BoardGame_Forge_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI BoardGame Forge

A simple web app that turns any theme into a mini card game with AI-generated
artwork. Built with Gradio.

Enter a theme such as "Space Cats", "Ancient Pirates", or "Escaping a
Corporate Office", or pick one of the provided examples, and the app
generates:

- A game title, description, and simple rules
- 4 unique cards (name, type, ability)
- A designed trading-card image for each card, using free AI-generated artwork

### How it works
- Game text (title, description, rules, cards) is generated locally with a
  procedural generator: Python code that combines the theme with pre-built
  word banks and templates.
- Card artwork is generated using Pollinations.ai, a free image generation
  service that requires no API key or account.
- Each card image is composed with Pillow (PIL) into a simple trading-card
  layout: a frame, title bar, type badge, and effect text box.
- The whole thing is wrapped in a Gradio interface, which is how it becomes
  a website with its own link instead of just notebook cells.

### Notebook sections
1. Setup
2. Game text generator
3. AI artwork generator
4. Card designer
5. Full pipeline
6. Gradio web app


## 1. Setup

All libraries used here are free. Gradio needs to be installed manually in
Colab; the rest are usually already available.


In [1]:
!pip install -q gradio


## 2. Game Text Generator

Instead of calling a paid AI model for the game text, this uses procedural
generation: Python randomly combines the theme with ready-made word banks
and sentence templates to build the title, description, rules, and cards.


In [2]:
import random

# Game title templates
TITLE_TEMPLATES = [
    "{theme} Wars",
    "The {theme} Chronicles",
    "Escape from {theme}",
    "{theme}: The Card Game",
    "Rise of the {theme}",
    "{theme} Showdown",
    "Legends of {theme}",
    "{theme} Uprising",
]

# Game description templates
DESCRIPTION_TEMPLATES = [
    "In this game, players take on the role of characters caught up in a {theme} scenario. Each player uses cards to outsmart their opponents and be the first to win.",
    "Players compete in a world inspired by {theme}, drawing and playing cards to gain the upper hand. Strategy and a little luck decide the winner.",
    "Set in a {theme} world, each player builds a small hand of powerful cards and battles to reach the win condition first.",
]

# Game rules template
RULES_TEMPLATE = """1. Each player starts with a hand of 4 cards.
2. Players take turns playing one card at a time.
3. When a card is played, follow the effect written on it.
4. The game ends when the deck runs out or a player reaches the target score.
5. The player with the most points, or the last one standing, wins the {theme} showdown."""

# Card types
CARD_TYPES = ["Attack", "Defense", "Support", "Trick", "Special", "Boost", "Sabotage"]

# Words combined with the theme to form a card name
NAME_PARTS = [
    "Guardian", "Blade", "Shadow", "Spark", "Wanderer", "Sentinel",
    "Rebel", "Trickster", "Champion", "Outcast", "Phantom", "Rogue"
]

# Card ability/effect pool
EFFECT_POOL = [
    "Draw 2 extra cards this turn.",
    "Block the next attack played against you.",
    "Steal one card from an opponent's hand.",
    "Skip your opponent's next turn.",
    "Double your score this round.",
    "Swap this card with any card in the discard pile.",
    "Force all players to reveal their hand.",
    "Gain +3 power for the rest of the game.",
    "Discard your hand and draw a fresh one.",
    "Copy the effect of the last card played.",
]


def generate_game(theme):
    """
    Builds a full mini card game (title, description, rules, 4 cards)
    from the given theme using templates and word banks.
    """
    theme_clean = theme.strip().title()

    title = random.choice(TITLE_TEMPLATES).format(theme=theme_clean)
    description = random.choice(DESCRIPTION_TEMPLATES).format(theme=theme_clean)
    rules = RULES_TEMPLATE.format(theme=theme_clean)

    chosen_types = random.sample(CARD_TYPES, 4)
    cards = []
    for card_type in chosen_types:
        name = f"{theme_clean} {random.choice(NAME_PARTS)}"
        effect = random.choice(EFFECT_POOL)
        cards.append({"name": name, "type": card_type, "effect": effect})

    return {"title": title, "description": description, "rules": rules, "cards": cards}


print("Game text generator ready.")


Game text generator ready.


## 3. AI Artwork Generator

For each card, a short text prompt is built from the theme, the card name,
and its type, then sent to Pollinations.ai to generate an image. If the
request fails for any reason, a simple placeholder image is used instead
so the app does not crash.


In [3]:
import requests
import urllib.parse
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont

# Color theme for each card type, used for both placeholders and card frames
TYPE_COLORS = {
    "Attack":   {"bg": (178, 34, 34),  "border": (255, 215, 0)},
    "Defense":  {"bg": (30, 60, 114),  "border": (192, 192, 192)},
    "Support":  {"bg": (34, 87, 47),   "border": (200, 255, 200)},
    "Trick":    {"bg": (75, 0, 130),   "border": (218, 165, 32)},
    "Special":  {"bg": (139, 69, 19),  "border": (255, 223, 0)},
    "Boost":    {"bg": (204, 85, 0),   "border": (255, 255, 255)},
    "Sabotage": {"bg": (20, 20, 20),   "border": (139, 0, 0)},
}


def get_font(size, bold=True):
    """Loads a font that ships with Colab/Ubuntu. Falls back to a basic font if missing."""
    try:
        path = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold \
            else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
        return ImageFont.truetype(path, size)
    except Exception:
        return ImageFont.load_default()


def make_placeholder_art(card):
    """A simple backup image used only if the AI art request fails."""
    colors = TYPE_COLORS.get(card["type"], {"bg": (100, 100, 100)})
    img = Image.new("RGB", (512, 512), colors["bg"])
    draw = ImageDraw.Draw(img)
    font = get_font(40)
    draw.text((40, 236), card["type"] + " Card", font=font, fill=(255, 255, 255))
    return img


def generate_card_art(theme, card):
    """
    Builds an image prompt from the theme and card info, and fetches free
    AI art from Pollinations.ai. No API key is required.
    """
    prompt = (
        f"{theme} {card['name']}, {card['type']} character, "
        f"fantasy trading card illustration, digital painting, detailed, vibrant colors, centered"
    )
    encoded_prompt = urllib.parse.quote(prompt)
    seed = random.randint(1, 999999)
    url = f"https://image.pollinations.ai/prompt/{encoded_prompt}?width=512&height=512&seed={seed}&nologo=true"

    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        return Image.open(BytesIO(response.content)).convert("RGB")
    except Exception as error:
        print(f"Could not fetch AI art for '{card['name']}', using a placeholder instead. Reason: {error}")
        return make_placeholder_art(card)


print("Art generator ready.")


Art generator ready.


## 4. Card Designer

This turns one card's data and its artwork into a finished trading-card
image, using Pillow to draw a frame, title bar, type badge, and effect text
box.


In [4]:
def wrap_text(text, font, max_width, draw):
    """Breaks a string into multiple lines so it fits inside max_width pixels."""
    words = text.split()
    lines = []
    current = ""
    for word in words:
        test_line = (current + " " + word).strip()
        box = draw.textbbox((0, 0), test_line, font=font)
        if box[2] - box[0] <= max_width:
            current = test_line
        else:
            if current:
                lines.append(current)
            current = word
    if current:
        lines.append(current)
    return lines


def draw_card(card, art_image, theme):
    """
    Composes the final card image: background color, artwork, title bar,
    type badge, and effect text, based on the card's type.
    """
    width, height = 500, 700
    colors = TYPE_COLORS.get(card["type"], {"bg": (60, 60, 60), "border": (255, 255, 255)})

    card_img = Image.new("RGB", (width, height), colors["bg"])
    draw = ImageDraw.Draw(card_img)

    # Outer border
    draw.rectangle([0, 0, width - 1, height - 1], outline=colors["border"], width=10)

    # Title bar
    title_font = get_font(28)
    draw.rectangle([15, 15, width - 15, 80], fill=(0, 0, 0))
    title_box = draw.textbbox((0, 0), card["name"], font=title_font)
    title_x = (width - (title_box[2] - title_box[0])) / 2
    draw.text((title_x, 25), card["name"], font=title_font, fill=(255, 255, 255))

    # Artwork
    art_box = (20, 90, width - 20, 430)
    art_resized = art_image.resize((art_box[2] - art_box[0], art_box[3] - art_box[1]))
    card_img.paste(art_resized, (art_box[0], art_box[1]))
    draw.rectangle(art_box, outline=colors["border"], width=4)

    # Type badge
    type_font = get_font(20)
    draw.rectangle([20, 440, 200, 480], fill=colors["border"])
    draw.text((30, 448), card["type"], font=type_font, fill=(0, 0, 0))

    # Effect text box
    effect_font = get_font(18, bold=False)
    draw.rectangle([20, 500, width - 20, height - 40], fill=(255, 255, 255))
    lines = wrap_text(card["effect"], effect_font, width - 60, draw)
    y = 515
    for line in lines:
        draw.text((35, y), line, font=effect_font, fill=(0, 0, 0))
        y += 24

    # Theme footer
    footer_font = get_font(13, bold=False)
    draw.text((25, height - 30), theme.title(), font=footer_font, fill=colors["border"])

    return card_img


print("Card designer ready.")


Card designer ready.


## 5. Full Pipeline

This combines everything: generate the game text, then generate and design
a card image for each of the 4 cards.


In [5]:
def forge_full_game(theme):
    """
    Full pipeline: theme -> game text -> AI artwork -> designed card images.
    Returns (game_data, list_of_card_images).
    """
    print(f"Generating game for theme: {theme}")
    game_data = generate_game(theme)
    print(f"Game created: {game_data['title']}")

    card_images = []
    for i, card in enumerate(game_data["cards"], start=1):
        print(f"Generating artwork for card {i} of 4: {card['name']}")
        art = generate_card_art(theme, card)
        card_img = draw_card(card, art, theme)
        card_images.append(card_img)

    print("Finished generating game and cards.")
    return game_data, card_images


print("Pipeline ready.")


Pipeline ready.


## 6. Gradio Web App

This section builds the actual website. Running the cell below starts a
Gradio interface and prints a public link you can open in any browser.

The interface has:
- A text box where a theme can be typed
- A list of example themes to choose from instead of typing
- A button to generate the game
- A text area showing the title, description, and rules
- A gallery showing the 4 generated card images


In [6]:
import gradio as gr

EXAMPLE_THEMES = [
    "Space Cats",
    "Ancient Pirates",
    "Escaping a Corporate Office",
    "Medieval Kingdom",
    "Desert Survivors",
    "Robot Uprising",
]


def build_game(theme):
    """
    Function connected to the Generate button. Takes the theme text,
    runs the full pipeline, and returns the text output and the
    list of card images for the Gradio interface.
    """
    if not theme or not theme.strip():
        return "Please enter a theme first.", []

    game_data, card_images = forge_full_game(theme)

    info_text = (
        f"## {game_data['title']}\n\n"
        f"**Description:** {game_data['description']}\n\n"
        f"**Rules:**\n\n{game_data['rules']}"
    )

    gallery_items = [
        (img, card["name"]) for img, card in zip(card_images, game_data["cards"])
    ]

    return info_text, gallery_items


with gr.Blocks(title="AI BoardGame Forge") as demo:
    gr.Markdown("# AI BoardGame Forge")
    gr.Markdown(
        "Enter a theme, or select one of the example themes below, "
        "then click Generate Game to create a mini card game with AI-generated artwork."
    )

    theme_input = gr.Textbox(
        label="Theme",
        placeholder="e.g. Space Cats, Ancient Pirates, Escaping a Corporate Office"
    )

    gr.Examples(
        examples=EXAMPLE_THEMES,
        inputs=theme_input,
        label="Example themes"
    )

    generate_button = gr.Button("Generate Game")

    game_info_output = gr.Markdown()
    card_gallery_output = gr.Gallery(
        label="Generated Cards",
        columns=4,
        height="auto"
    )

    generate_button.click(
        fn=build_game,
        inputs=theme_input,
        outputs=[game_info_output, card_gallery_output]
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://014622c6bb1939da09.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---

## Project Summary

- Idea: turn any theme into a mini card game with AI-generated artwork,
  delivered as a web app instead of plain notebook output
- Tech used:
  - Python and random for procedural text generation
  - Pollinations.ai for free, no-key AI image generation
  - Pillow (PIL) for composing each card into a trading-card layout
  - Gradio for the web interface and the shareable link
- Why this stack: every part is free with no signup or API key, works
  inside Google Colab, and produces a real link that can be opened in a
  browser during a presentation instead of only running inside the notebook
- Fallback design: if the art service is slow or unavailable, a themed
  placeholder image is used automatically so the app does not crash
- Possible future improvements: let users choose the number of cards, add
  a button to regenerate only the artwork, allow downloading the cards as
  a single image or PDF, add card rarity levels with different border styles
